<a href="https://colab.research.google.com/github/manikandansakthivel175-cmd/E-COMMERCE_RAG_ASSISTANT/blob/main/Ecommerce_RAG_Assistant_Staged.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 5 — AI Agents & RAG
## E-commerce RAG Assistant

### Hands-on project

We will build the project **feature by feature**.

**Feature 1:** Product catalog + document ingestion  
**Feature 2:** Semantic product retrieval (RAG)  
**Feature 3:** Grounded shopping assistant  
**Feature 4:** Conversation memory + observability

**Final flow:** User Question → Retrieve Products → Build Context → LLM Answer


## Teaching structure

For every feature we clearly identify:

**Input → Tool/Function → Processing → Output**

The main goal is to demonstrate how **Retrieval-Augmented Generation (RAG)** gives the LLM relevant product information before generating an answer.


## Colab setup

For faster generation, select:

**Runtime → Change runtime type → T4 GPU**

CPU can also work, but the local LLM will be slower.


In [1]:
!pip -q install -U transformers accelerate sentencepiece requests sentence-transformers faiss-cpu pandas
print("✅ Installation complete")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 88.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 w

# 1. Initialize the AI Model

The language model understands the user's question and generates the final response.

We use a small local instruction-following model so the project does not require a paid API key.


In [2]:
import json
import re
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model_kwargs = {}
if device == "cuda":
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    **model_kwargs
).to(device)

model.eval()
print("✅ Model loaded:", MODEL_NAME)


Device: cuda


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [3]:
def ask_llm(user_prompt, system_prompt="You are a helpful e-commerce shopping assistant."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

print("✅ LLM helper ready")


✅ LLM helper ready


# 2. Create the E-commerce Product Catalog

For this classroom project, we create a small product catalog.

In a real application, this data can come from a database, CSV file, product API, or uploaded product documents.


In [4]:
products = [
    {
        "id": "P001",
        "name": "NoiseFit Smart Watch",
        "category": "Wearables",
        "price": 2499,
        "rating": 4.3,
        "stock": 18,
        "description": "Bluetooth smartwatch with fitness tracking, heart-rate monitoring, sleep tracking and 1.78-inch display.",
        "features": "Fitness tracking, heart-rate monitoring, sleep tracking, Bluetooth calling",
        "ideal_for": "Fitness users and students"
    },
    {
        "id": "P002",
        "name": "Boat Airdopes Wireless Earbuds",
        "category": "Audio",
        "price": 1299,
        "rating": 4.1,
        "stock": 35,
        "description": "True wireless earbuds with Bluetooth connectivity, low-latency mode and charging case.",
        "features": "Wireless audio, low latency, Bluetooth, charging case",
        "ideal_for": "Music, calls and casual gaming"
    },
    {
        "id": "P003",
        "name": "HP 15s Laptop",
        "category": "Laptops",
        "price": 48999,
        "rating": 4.4,
        "stock": 9,
        "description": "15.6-inch laptop suitable for programming, office work and everyday productivity.",
        "features": "15.6-inch display, SSD storage, Wi-Fi, programming friendly",
        "ideal_for": "Students, programmers and office users"
    },
    {
        "id": "P004",
        "name": "Logitech K380 Keyboard",
        "category": "Accessories",
        "price": 2999,
        "rating": 4.5,
        "stock": 22,
        "description": "Compact wireless keyboard designed for comfortable typing across multiple devices.",
        "features": "Wireless, compact, multi-device support",
        "ideal_for": "Students and professionals"
    },
    {
        "id": "P005",
        "name": "Samsung Galaxy M15",
        "category": "Smartphones",
        "price": 14999,
        "rating": 4.2,
        "stock": 14,
        "description": "Budget smartphone with a large display, long-lasting battery and multi-camera setup.",
        "features": "Large display, long battery, multi-camera setup",
        "ideal_for": "Everyday smartphone users"
    },
    {
        "id": "P006",
        "name": "JBL Go Portable Speaker",
        "category": "Audio",
        "price": 1899,
        "rating": 4.4,
        "stock": 27,
        "description": "Compact portable Bluetooth speaker for music at home or outdoors.",
        "features": "Bluetooth, portable design, rechargeable battery",
        "ideal_for": "Music lovers and travel"
    }
]

df = pd.DataFrame(products)
df


,id,name,category,price,rating,stock,description,features,ideal_for
0,P001,NoiseFit Smart Watch,Wearables,2499,4.3,18,"Bluetooth smartwatch with fitness tracking, he...","Fitness tracking, heart-rate monitoring, sleep...",Fitness users and students
1,P002,Boat Airdopes Wireless Earbuds,Audio,1299,4.1,35,True wireless earbuds with Bluetooth connectiv...,"Wireless audio, low latency, Bluetooth, chargi...","Music, calls and casual gaming"
2,P003,HP 15s Laptop,Laptops,48999,4.4,9,"15.6-inch laptop suitable for programming, off...","15.6-inch display, SSD storage, Wi-Fi, program...","Students, programmers and office users"
3,P004,Logitech K380 Keyboard,Accessories,2999,4.5,22,Compact wireless keyboard designed for comfort...,"Wireless, compact, multi-device support",Students and professionals
4,P005,Samsung Galaxy M15,Smartphones,14999,4.2,14,"Budget smartphone with a large display, long-l...","Large display, long battery, multi-camera setup",Everyday smartphone users
5,P006,JBL Go Portable Speaker,Audio,1899,4.4,27,Compact portable Bluetooth speaker for music a...,"Bluetooth, portable design, rechargeable battery",Music lovers and travel


# 🚀 FEATURE 1 — Product Document Preparation

## Goal

Convert structured product records into searchable text documents.

**Input:** Product catalog  
**Output:** Product documents for retrieval


In [5]:
def product_to_document(product):
    return f"""
Product ID: {product['id']}
Product Name: {product['name']}
Category: {product['category']}
Price: ₹{product['price']}
Rating: {product['rating']}/5
Stock: {product['stock']}
Description: {product['description']}
Features: {product['features']}
Ideal For: {product['ideal_for']}
""".strip()

documents = [product_to_document(p) for p in products]

for doc in documents[:2]:
    print("=" * 60)
    print(doc)

print("✅ Product documents created:", len(documents))


Product ID: P001
Product Name: NoiseFit Smart Watch
Category: Wearables
Price: ₹2499
Rating: 4.3/5
Stock: 18
Description: Bluetooth smartwatch with fitness tracking, heart-rate monitoring, sleep tracking and 1.78-inch display.
Features: Fitness tracking, heart-rate monitoring, sleep tracking, Bluetooth calling
Ideal For: Fitness users and students
Product ID: P002
Product Name: Boat Airdopes Wireless Earbuds
Category: Audio
Price: ₹1299
Rating: 4.1/5
Stock: 35
Description: True wireless earbuds with Bluetooth connectivity, low-latency mode and charging case.
Features: Wireless audio, low latency, Bluetooth, charging case
Ideal For: Music, calls and casual gaming
✅ Product documents created: 6


# 🚀 FEATURE 2 — Semantic Retrieval

The RAG system first retrieves the most relevant product documents.

Flow:

```text
User Question
      ↓
Embedding Model
      ↓
Vector Search
      ↓
Relevant Products
```

This is the **Retrieval** part of RAG.


In [6]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL)

document_embeddings = embedder.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

dimension = document_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(document_embeddings.astype("float32"))

print("Embedding dimension:", dimension)
print("Indexed documents:", index.ntotal)
print("✅ Vector index ready")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384
Indexed documents: 6
✅ Vector index ready


In [7]:
def retrieve_products(query, top_k=3):
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "product": products[int(idx)],
            "document": documents[int(idx)]
        })

    return results

query = "I need a budget wireless audio product for music"
results = retrieve_products(query, top_k=3)

print("QUERY:", query)
print("\nRETRIEVED PRODUCTS")
for r in results:
    print(f"Score: {r['score']:.3f} | {r['product']['name']}")


QUERY: I need a budget wireless audio product for music

RETRIEVED PRODUCTS
Score: 0.452 | JBL Go Portable Speaker
Score: 0.424 | Boat Airdopes Wireless Earbuds
Score: 0.310 | Logitech K380 Keyboard


## Feature 2 — Understand Input and Output

```text
INPUT
User question
   ↓
Embedding Model
   ↓
Vector Database / FAISS
   ↓
OUTPUT
Relevant product documents
```

The vector search does not generate the final answer. It only finds useful context.

**Feature 2 complete ✅**


# 🚀 FEATURE 3 — Grounded E-commerce RAG Assistant

Now we connect retrieval with the LLM.

The LLM receives:

**User question + Retrieved product context**

Important rule:

- Answer using the supplied catalog context.
- Do not invent price, stock, rating or features.
- If the catalog does not contain the answer, say that it is not available in the catalog.


In [8]:
def build_context(results):
    return "\n\n".join(
        f"[Product {i+1}]\n{r['document']}"
        for i, r in enumerate(results)
    )

def ecommerce_rag_answer(user_query, top_k=3):
    retrieved = retrieve_products(user_query, top_k=top_k)
    context = build_context(retrieved)

    prompt = f"""
You are an e-commerce RAG assistant.

USER QUESTION:
{user_query}

RETRIEVED PRODUCT CONTEXT:
{context}

INSTRUCTIONS:
- Answer only from the retrieved product context.
- Recommend products only when supported by the context.
- Do not invent price, stock, rating, features, delivery details or policies.
- If the requested information is not present, clearly say it is not available in the catalog.
- Keep the answer concise and useful.
- Mention the product name and price when relevant.
"""

    answer = ask_llm(prompt)

    return {
        "query": user_query,
        "retrieved": retrieved,
        "context": context,
        "answer": answer
    }

rag_result = ecommerce_rag_answer(
    "Suggest a good wireless audio product under ₹2000"
)

print("QUESTION")
print(rag_result["query"])

print("\nANSWER")
print("=" * 60)
print(rag_result["answer"])


QUESTION
Suggest a good wireless audio product under ₹2000

ANSWER
Based on your criteria of being under ₹2000, I recommend the **Boat Airdopes Wireless Earbuds** (P002). They offer true wireless audio, low latency, and a charging case, making them ideal for music, calls, and casual gaming. The product has a great rating of 4.1/5 and is currently in stock.


# 4. Test Different Customer Questions

Examples:

- Suggest a laptop for programming.
- Show me a wireless audio product under ₹2000.
- Which product has the highest rating?
- I need something for students.
- How much is the smartwatch?


In [9]:
test_questions = [
    "Suggest a laptop for programming",
    "Show me a wireless audio product under ₹2000",
    "Which product is useful for students?",
    "How much is the smartwatch?"
]

for q in test_questions:
    result = ecommerce_rag_answer(q, top_k=3)
    print("\n" + "=" * 70)
    print("USER:", q)
    print("ASSISTANT:", result["answer"])



USER: Suggest a laptop for programming
ASSISTANT: Based on your requirements for a laptop suitable for programming, I recommend the **HP 15s Laptop** (Product ID: P003). It has a 15.6-inch display, SSD storage, Wi-Fi connectivity, and is programmed-friendly, making it ideal for students, programmers, and office users. The current price is ₹48,999.

USER: Show me a wireless audio product under ₹2000
ASSISTANT: Based on your criteria of being under ₹2000, I recommend:

**JBL Go Portable Speaker**
Product ID: P006  
Price: ₹1899  
Ideal For: Music lovers and travel

USER: Which product is useful for students?
ASSISTANT: Based on the provided context, Product 1 (HP 15s Laptop) is useful for students as it is suitable for programming, office work, and everyday productivity. It has a 15.6-inch display, SSD storage, Wi-Fi, and is programmed-friendly, making it ideal for students, programmers, and office users.

USER: How much is the smartwatch?
ASSISTANT: The NoiseFit Smart Watch costs ₹2499

# 5. Conversation Memory

Memory allows the assistant to retain useful preferences between turns.

For this classroom project, we use a simple Python dictionary.

Example:

```text
User: I am a student.
Assistant: ...
User: Suggest a laptop.
Assistant: Uses the student preference + retrieved products.
```


In [10]:
agent_memory = {
    "user_preferences": {},
    "conversation": []
}

def remember_preference(key, value):
    agent_memory["user_preferences"][key] = value

def memory_text():
    if not agent_memory["user_preferences"]:
        return "No saved preferences."
    return json.dumps(agent_memory["user_preferences"], indent=2)

remember_preference("user_type", "student")

print("MEMORY")
print(memory_text())


MEMORY
{
  "user_type": "student"
}


# 6. RAG Assistant with Memory

Now the assistant uses both:

**Memory + Retrieved Product Context + User Question**


In [11]:
def ecommerce_rag_with_memory(user_query, top_k=3):
    retrieved = retrieve_products(user_query, top_k=top_k)
    context = build_context(retrieved)

    prompt = f"""
You are a helpful e-commerce RAG assistant.

USER MEMORY:
{memory_text()}

USER QUESTION:
{user_query}

RETRIEVED PRODUCT CONTEXT:
{context}

RULES:
- Use memory only when it is relevant.
- Answer only using the retrieved catalog context for product facts.
- Do not invent product facts.
- If information is missing, say so.
- Keep the response concise.
"""

    answer = ask_llm(prompt)

    agent_memory["conversation"].append({
        "user": user_query,
        "assistant": answer
    })

    return answer

print(ecommerce_rag_with_memory("Suggest a laptop suitable for me"))


Based on your requirements as a student, I would recommend the **HP 15s Laptop** (Product ID: P003). It's suitable for programming, office work, and everyday productivity, featuring a 15.6-inch display, SSD storage, Wi-Fi connectivity, and is rated highly by customers.


# 7. Human-in-the-loop

A human can review an AI recommendation before an external action such as placing an order.

This adds a safety and approval checkpoint.

For this demo, we only approve or reject the generated recommendation.


In [ ]:
result = ecommerce_rag_answer(
    "Recommend a product for music under ₹2000",
    top_k=3
)

print("GENERATED RECOMMENDATION")
print("=" * 60)
print(result["answer"])

approval = input(
    "\nApprove this recommendation? (yes/no): "
).strip().lower()

if approval == "yes":
    print("✅ Human approved the recommendation.")
else:
    print("🛑 Human rejected the recommendation.")


GENERATED RECOMMENDATION
Based on your criteria of recommending a product for music under ₹2000, I would recommend:

JBL Go Portable Speaker (P006)
Price: ₹1899


# 8. Agent Observability

Observability means seeing what happened during execution.

We record:

1. User query
2. Retrieval step
3. Number of documents retrieved
4. LLM generation step
5. Final status


In [ ]:
def run_with_trace(user_query, top_k=3):
    trace = []

    trace.append({
        "step": 1,
        "operation": "user_query",
        "status": "received"
    })

    retrieved = retrieve_products(user_query, top_k=top_k)

    trace.append({
        "step": 2,
        "operation": "semantic_retrieval",
        "status": "completed",
        "documents_retrieved": len(retrieved)
    })

    context = build_context(retrieved)

    prompt = f"""
You are an e-commerce RAG assistant.

Question:
{user_query}

Catalog context:
{context}

Answer only from the context. Do not invent product facts.
"""

    answer = ask_llm(prompt)

    trace.append({
        "step": 3,
        "operation": "llm_generation",
        "status": "completed"
    })

    trace.append({
        "step": 4,
        "operation": "final_response",
        "status": "completed"
    })

    return answer, trace

answer, execution_trace = run_with_trace(
    "What is a good compact wireless keyboard?"
)

print("ANSWER")
print(answer)

print("\nAGENT EXECUTION TRACE")
print("=" * 60)

for item in execution_trace:
    print(item)


# 9. Multi-Agent Architecture

The working implementation can remain as **one RAG assistant**.

A future multi-agent architecture could divide responsibilities:

```text
                 Manager Agent
                       |
          +------------+-------------+
          |            |             |
          ▼            ▼             ▼
     Retrieval      Product       Support
       Agent        Agent          Agent
          |
          ▼
      Vector DB
```

- **Manager Agent:** coordinates the workflow.
- **Retrieval Agent:** finds relevant products.
- **Product Agent:** explains product information.
- **Support Agent:** handles catalog-supported questions about policies or issues.


# 🔗 Complete E-commerce RAG Flow

```text
                         USER
                           |
                           ▼
                    USER QUESTION
                           |
                           ▼
                    QUERY EMBEDDING
                           |
                           ▼
                 VECTOR SEARCH / FAISS
                           |
                           ▼
                 RELEVANT PRODUCTS
                           |
                           ▼
              RETRIEVED PRODUCT CONTEXT
                           |
              +------------+------------+
              |                         |
              ▼                         ▼
        USER MEMORY                 LLM MODEL
              |                         |
              +------------+------------+
                           |
                           ▼
                    GROUNDED ANSWER
                           |
                           ▼
                     USER RESPONSE
```

**One project → RAG retrieval → grounded generation → memory → observability → agent concepts.**


# 🎯 Final Project Summary

### 1. Product Document Preparation
**Input:** Product catalog  
**Output:** Searchable product documents

### 2. Semantic Retrieval
**Input:** User question  
**Output:** Relevant product documents

### 3. RAG Answer Generation
**Input:** User question + retrieved context  
**Output:** Grounded shopping response

### 4. Memory
Stores useful user preferences and conversation history.

### 5. Human-in-the-loop
Allows a person to approve or reject an AI-generated recommendation.

### 6. Observability
Tracks retrieval and generation steps.

### Technologies

| Component | Technology |
|---|---|
| Language | Python |
| LLM | Qwen2.5-1.5B-Instruct |
| Embeddings | Sentence Transformers |
| Vector Search | FAISS |
| Data | Pandas / Python records |
| Environment | Google Colab |
| Architecture | RAG |
